In [ ]:
#done and updated by Jingyi
# on 26/04/2025
#done and updated by LLZ
# on 28/02/2022

# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os

print("Running SA SAMA Web Scraping Tool v.1.0")

now=datetime.datetime.now()
filename= 'SA SAMA SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

scriptfolder=os.path.dirname(os.path.abspath(__file__))
os.chdir(scriptfolder)

regdict={'SA SAMA 3': 'https://www.sama.gov.sa/en-US/LicenseEntities/Pages/LicensedBanks.aspx',
         'SA SAMA 4': 'https://www.sama.gov.sa/en-US/LicenseEntities/Pages/FinanceLicencedEntities.aspx',
         'SA SAMA 5': 'https://www.sama.gov.sa/en-US/LicenseEntities/Pages/Licensed_Companies.aspx',
         'SA SAMA 6': 'https://www.sama.gov.sa/en-US/LicenseEntities/Pages/Licensed_Payment_Service_Providers_companies.aspx',
         'SA SAMA 7': 'https://www.sama.gov.sa/en-US/LicenseEntities/Pages/Licensed-Money-Exchangers-Centers.aspx'}
#Removed/outdated
#'SA SAMA 1': 'https://www.sama.gov.sa/en-US/License/Pages/SaudiBanks.aspx', 
#'SA SAMA 2': 'https://www.sama.gov.sa/en-US/License/Pages/InternationalBanks.aspx',

driver = webdriver.Chrome()
driver.maximize_window()

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')

for reg in regdict:
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(3)
    soup=BeautifulSoup(driver.page_source, 'html.parser')
    pag_div = soup.find('div', {'class': 'footable-pagination-wrapper'})
    if pag_div is None:
        pages = 1
    else:
        pages = int(pag_div.text.strip().split()[-1])
    for page in range(pages):
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find('table', id='tblPermittedFintech')
        headers = table.find('thead').find_all('tr')[-1]
        headers = [ele.text.strip() for ele in headers]
        trs = table.find('tbody').find_all('tr')
        name_ix = False
        id_1 = False
        id_2 = False
        license_status =False
        if 'Entity Name' in headers:
            name_ix = headers.index('Entity Name')
        elif 'Company Name' in headers:
            name_ix = headers.index('Company Name')
        else:
            for i, hea in enumerate(headers):
                if 'name' in hea.lower():
                    name_ix = i
                    break
            else:
                raise Exception()
        if 'Unified Number' in headers:
            id_1 = ['Unified Number', headers.index('Unified Number')]
        elif 'ID Number' in headers:
            id_1 = ['ID Number', headers.index('ID Number')]
        if 'Commercial Register Number' in headers:
            id_2 = ['Commercial Register Number', headers.index('Commercial Register Number')]
        elif 'ID Licensing Number' in headers:
            id_2 = ['Licensing Number', headers.index('Licensing Number')]
        for tr in trs:
            tds = tr.find_all('td')
            sqldict['ListProcessDate'].append(processdate)
            sqldict['Name'].append(tds[name_ix].text.strip())
            url = tds[name_ix].find('a', href=True) 
            if url:
                sqldict['Website'].append(url['href'])
            if id_1:
                sqldict['InternalID_1_type'].append(id_1[0])
                sqldict['InternalID_1'].append(tds[id_1[1]].text.strip())
            if id_2:
                sqldict['InternalID_2_type'].append(id_2[0])
                sqldict['InternalID_2'].append(tds[id_2[1]].text.strip())
            sqldict['Cntry'].append('SA')
            sqldict['RegCtry'].append('SA')
            sqldict['RegCode'].append('SAMA')
            sqldict['ListCode'].append(reg.split()[-1])
            sqldict['RegulationType'].append('Regulated')##depends
            for key in sqldict:
                if len(sqldict['ListProcessDate'])>len(sqldict[key]):
                    sqldict[key].append('')
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        if soup.find('li', {'data-page':str(page+2)}):
            next_button = driver.find_element(By.XPATH, f"//li[@data-page='{page+2}']/a")
        driver.execute_script("arguments[0].click();", next_button)
        sleep(1.5)

df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()

sleep(3)

driver.quit()
    
    